💡 **Environment:** `clamp-analyses`

# Supplementary Figure 7: Cytokine gene-loading comparisons

Compares the gene loadings of the IFN-alpha mechanisms recovered by both the ARCHS4 consensus projection and the model trained on the cytokine dataset (GSE133218).

Each model panel labels the 10 highest-loading pathway genes available in the plotted top 1% of loadings (or all available genes if fewer than 10), using evenly spaced labels and connector lines.


In [ ]:
source(here::here("scripts/panels/style.R"))
suppressPackageStartupMessages({library(ggrepel); library(readr); library(dplyr)})
source(here("scripts/archs4/projections/plots.R"))
source(here("scripts/panels/gm_ids.R"))
set.seed(123)
out <- here("output/99_panels/supp7/source_data")
dir.create(out, recursive=TRUE, showWarnings=FALSE)
agg_dir <- here("output/03_model_biology/02_archs4/03_projections/aggregate")
prod_root <- here("output/01_model_building/02_archs4/03_projections")
pathway_file <- function(name) {
  candidates <- here("data", "pathways", name)
  found <- candidates[file.exists(candidates)]
  if (!length(found)) stop("Missing pathway input: ", name)
  found[1]
}

## Cytokine source data

In [ ]:
cyt_dataset <- "cyt_ifna_GSE133218"
hits <- fread(file.path(agg_dir, 'lv_pathway_hits_long.csv'))[dataset == cyt_dataset]
setnames(hits, 'p.adjust', 'fdr', skip_absent = TRUE)

curated <- fread(here("workflow/config/cytokine_mechanisms.csv"), na.strings = "")
manual_recovery <- merge(curated[, .(mechanism, category, model, LV = module, term = gene_set, supp7_example)],
                         hits[, .(model, LV, term, fdr, database)], by = c("model", "LV", "term"), all.x = TRUE, sort = FALSE)
manual_recovery[, recovered := !is.na(LV)]
stopifnot(manual_recovery[recovered == TRUE, !anyNA(fdr) & all(fdr < 0.05)])
manual_recovery[, mechanism := factor(mechanism, levels = unique(curated$mechanism))]
setorder(manual_recovery, mechanism, model)

print(manual_recovery[, .(mechanism, model, LV, pathway = term, fdr, recovered)])

source(here("scripts", "archs4", "common.R"))

read_gmt_sets <- function(path) {
  x <- strsplit(readLines(path, warn = FALSE), "\t", fixed = TRUE)
  out <- lapply(x, function(row) unique(row[-c(1L, 2L)]))
  names(out) <- vapply(x, `[[`, "", 1L)
  out
}
gene_sets <- list(
  canonical = read_gmt_sets(pathway_file("c2.cp.v2026.1.Hs.symbols.gmt")),
  hallmark  = read_gmt_sets(pathway_file("h.all.v2026.1.Hs.symbols.gmt"))
)

z_registry <- fread(file.path(prod_root, cyt_dataset, "mechanism_models.tsv"))
z_mats <- list(
  ARCHS4 = read_matrix_csv(here("output", "98_final_models", "consensus", "archs4", "Z.csv")),
  local  = read_matrix_csv(here(z_registry[model == "local"]$z))
)

top_pct <- 0.01
recovered_rows <- manual_recovery[recovered == TRUE]

gene_loadings <- rbindlist(lapply(seq_len(nrow(recovered_rows)), function(i) {
  r <- recovered_rows[i]
  z <- z_mats[[r$model]]
  members <- intersect(gene_sets[[r$database]][[r$term]], rownames(z))
  n_top <- max(1L, ceiling(nrow(z) * top_pct))
  ord <- order(z[, r$LV], decreasing = TRUE)[seq_len(n_top)]
  data.table(dataset = cyt_dataset, comparison_id = as.character(r$mechanism), model = r$model,
             rank = seq_along(ord), gene = rownames(z)[ord], loading = z[ord, r$LV],
             is_gene_set = rownames(z)[ord] %chin% members, in_top_loading_set = TRUE,
             n_gene_set_in_universe = length(members))
}))

comparisons <- copy(manual_recovery)
setnames(comparisons, "term", "gene_set")
comparisons[, `:=`(dataset = cyt_dataset, comparison_id = as.character(mechanism), top_pct = top_pct)]
comparisons <- merge(
  comparisons,
  gene_loadings[, .(n_gene_set_in_top_loading = sum(is_gene_set),
                     n_gene_set_in_universe = max(n_gene_set_in_universe)),
                by = .(comparison_id, model)],
  by = c("comparison_id", "model"), all.x = TRUE)
comparisons[is.na(n_gene_set_in_top_loading), n_gene_set_in_top_loading := 0L]
comparisons[is.na(n_gene_set_in_universe), n_gene_set_in_universe := 0L]

comparison_tests <- comparisons[, {
  a <- .SD[model == "ARCHS4"]
  b <- .SD[model == "local"]
  p <- NA_real_
  arch_fraction <- NA_real_
  local_fraction <- NA_real_
  if (nrow(a) == 1L && nrow(b) == 1L && isTRUE(a$recovered) && isTRUE(b$recovered) &&
      identical(a$gene_set, b$gene_set) &&
      a$n_gene_set_in_universe > 0L && b$n_gene_set_in_universe > 0L) {
    arch_fraction <- a$n_gene_set_in_top_loading / a$n_gene_set_in_universe
    local_fraction <- b$n_gene_set_in_top_loading / b$n_gene_set_in_universe
    p <- fisher.test(matrix(c(a$n_gene_set_in_top_loading,
                              a$n_gene_set_in_universe - a$n_gene_set_in_top_loading,
                              b$n_gene_set_in_top_loading,
                              b$n_gene_set_in_universe - b$n_gene_set_in_top_loading),
                            nrow = 2, byrow = TRUE))$p.value
  }
  .(p_value = p, arch_fraction = arch_fraction, local_fraction = local_fraction)
}, by = comparison_id]
comparison_tests[, p_adj := p.adjust(p_value, method = "BH")]
comparisons <- merge(comparisons, comparison_tests, by = "comparison_id", all.x = TRUE)

comparisons[, comparison_note := ""]
comparisons[model == "local" & !recovered, comparison_note := "Not recovered"]
comparisons[recovered == TRUE & is.na(p_adj), comparison_note := "Different matched pathway"]
comparisons[recovered == TRUE & !is.na(p_adj), comparison_note := paste0(
  fifelse(arch_fraction > local_fraction, "ARCHS4 > Cyt model",
          fifelse(arch_fraction < local_fraction, "Cyt model > ARCHS4", "equal fraction")),
  " (FDR ", formatC(p_adj, format = "e", digits = 1), ")")]

for (name in c("manual_recovery", "gene_loadings", "comparisons"))
  fwrite(get(name), file.path(out, paste0("00_cytokines_", name, ".csv")))
rm(z_mats); gc(verbose=FALSE)

In [ ]:
file.copy(file.path(agg_dir,"lv_stats_long.csv"), file.path(out,"lv_stats_long.csv"), overwrite=TRUE)
stopifnot(file.exists(file.path(out,"lv_stats_long.csv")))

## Assemble and export the figure

In [ ]:
out <- if (nzchar(Sys.getenv("SUPP7_SOURCE_DIR"))) Sys.getenv("SUPP7_SOURCE_DIR") else here("output/99_panels/supp7/source_data")
S <- list(fread_gm(file.path(out, "00_cytokines_comparisons.csv")))
L <- list(fread_gm(file.path(out, "00_cytokines_gene_loadings.csv")))
S[[1]][, LV := sub("^metaLV", "GM", LV)]

wide <- dcast(S[[1]], comparison_id + category ~ model,
  value.var=c("recovered", "LV", "gene_set", "fdr", "n_gene_set_in_top_loading", "n_gene_set_in_universe"))
examples <- wide[recovered_ARCHS4 == TRUE & recovered_local == TRUE &
                   gene_set_ARCHS4 == gene_set_local & fdr_ARCHS4 < .05 & fdr_local < .05]
examples <- examples[comparison_id %in% S[[1]][supp7_example == TRUE, comparison_id]]
examples[, mechanism_order:=match(comparison_id, unique(S[[1]]$comparison_id))]
examples <- unique(examples[order(mechanism_order)], by="gene_set_ARCHS4")
examples[, group:=1L]
stopifnot(nrow(examples) >= 1L)
fwrite(examples, file.path(out, "panel_examples.csv"))

short_fdr <- function(x) ifelse(is.na(x), "—", formatC(x, format="e", digits=1))
# RNA-seq dataset (with accession) in which each example was found, shown under each title.
dataset_label <- c(cyt_ifna_GSE133218="IFN-alpha-treated EndoC-betaH1 cells (GSE133218)")
example_model_plot <- function(x, model_name, xmax) {
  g <- x$group; mechanism_name <- x$comparison_id
  s <- copy(S[[g]][comparison_id == mechanism_name])
  d <- copy(L[[g]][comparison_id == mechanism_name & model == model_name])
  st <- s[model == model_name]
  gene_labels <- head(d[is_gene_set == TRUE][order(rank)], 10L)
  # Fixed, evenly spaced label positions prevent collisions, including tied loadings.
  loading_range <- range(d$loading, finite=TRUE)
  loading_span <- max(diff(loading_range), abs(loading_range), .Machine$double.eps)
  gene_labels[, label_y := seq(loading_range[2] + .20 * loading_span,
                               loading_range[1] + .10 * loading_span,
                               length.out=.N)]
  gene_labels[, label_x := .62 * xmax]
  ggplot(d, aes(rank, loading)) +
    geom_point(colour="grey78", size=.18, alpha=.35) +
    geom_point(data=d[is_gene_set==TRUE], colour=if (model_name == "ARCHS4") "#332288" else "#DDCC77", size=.55) +
    geom_segment(data=gene_labels, aes(xend=label_x - .02 * xmax, yend=label_y),
                 colour="grey55", linewidth=.15) +
    geom_text(data=gene_labels, aes(x=label_x, y=label_y, label=gene),
              hjust=0, size=5/.pt) +
    scale_y_continuous(expand=expansion(mult=c(.03,.30))) +
    scale_x_continuous(limits=c(0, xmax)) +
    labs(x=NULL, y="Loading") + nm_theme() +
    theme(axis.text=element_text(size=5), axis.title.y=element_text(size=5), plot.margin=margin(1,2,1,2,"mm"))
}
example_model_header <- function(x, model_name) {
  s <- S[[x$group]][comparison_id == x$comparison_id]
  st <- s[model == model_name]
  model_label <- if (model_name == "ARCHS4") "ARCHS4 projection" else proj_local_model_label(unique(s$dataset))
  h <- ggdraw() + draw_label(model_label, y=.8, vjust=0, size=5.5)
  stopifnot(nrow(st) == 1L, isTRUE(st$recovered[1]), st$fdr[1] < .05)
  e <- floor(log10(st$fdr[1])); m <- sprintf("%.1f", st$fdr[1] / 10^e)
  h + draw_label(sprintf("%s, %d/%d genes (%.0f%%)", st$LV[1], st$n_gene_set_in_top_loading[1], st$n_gene_set_in_universe[1], 100*st$n_gene_set_in_top_loading[1]/st$n_gene_set_in_universe[1]), y=.45, vjust=0, size=5) +
    draw_label(bquote("FDR" == .(m) %*% 10^displaystyle(.(e))), y=.06, vjust=0, size=5)
}
example_pair <- function(x) {
  g <- x$group; mechanism_name <- x$comparison_id
  s <- S[[g]][comparison_id == mechanism_name & model == "ARCHS4"]
  term <- proj_pretty_term(s$gene_set[1])
  # Each example is a direct side-by-side comparison at the same scale:
  # ARCHS4 | local.
  source_line <- dataset_label[[s$dataset[1]]]
  xmax <- max(L[[g]][comparison_id == mechanism_name, rank])
  plot_grid(ggdraw() + draw_label(stringr::str_wrap(term, width=42), y=.45, vjust=0, size=6, fontface="bold") +
              draw_label(source_line, y=.12, vjust=0, size=5, colour="black"),
            plot_grid(example_model_header(x, "ARCHS4"), example_model_header(x, "local"), nrow=1),
            plot_grid(example_model_plot(x, "ARCHS4", xmax), example_model_plot(x, "local", xmax), nrow=1, align="hv", axis="tblr"),
            ggdraw() + draw_label("Gene rank", size=5),
            ncol=1, rel_heights=c(.19,.19,.56,.06))
}
E <- plot_grid(plotlist=lapply(seq_len(nrow(examples)), function(i) example_pair(examples[i])), ncol=2)
nm_export(list(E), 7, "ARCHS4 projection versus the dataset-trained cytokine model: gene loadings of shared mechanisms",
          height=65 * ceiling(nrow(examples) / 2))
